# 06 — MEM Electricity | Automatic Download (raw landing)

Finds the newest **Informe de Desempeño — Anexos** workbook published by the Ministry of Energy and Mines (MEM)
and lands it, unchanged, in `/Volumes/rd_economic_intelligence/bronze/mem_raw/ede_performance/`.

| Step | Rule |
|---|---|
| Discover | MEM website media index (WordPress REST API `wp-json/wp/v2/media`), files whose name contains `Desempeno` and `Anexos` and end in `.xlsx`; newest by publication date |
| Skip | If a file with the same name, or the same size and content hash, is already in the Volume, nothing is downloaded |
| Validate | The download must open as a workbook with the sheets `EDE's` and `Nuevo Regimen tarifario`; otherwise the task fails and nothing is written |
| Land | Original file name kept; Bronze always reads the most recent file |
| Signal | Task value `downloaded` (true/false) lets the scheduled Job skip Bronze, Silver and Gold when nothing new was published |

Runs before `06_MEM_Electricity_Bronze` in the main Lakeflow Job and in the weekly job `rd_mem_electricity_weekly`. **Status:** Development.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import json, hashlib, io, re, urllib.request
import openpyxl

LANDING = "/Volumes/rd_economic_intelligence/bronze/mem_raw/ede_performance/"
API = "https://mem.gob.do/wp-json/wp/v2/media?search=Anexos&per_page=100&orderby=date&order=desc&_fields=date,source_url"
REQUIRED_SHEETS = {"EDE's", "Nuevo Regimen tarifario"}
HEADERS = {"User-Agent": "rd-economic-intelligence/1.0 (portfolio data pipeline)"}

def get(url, timeout=60):
    with urllib.request.urlopen(urllib.request.Request(url, headers=HEADERS), timeout=timeout) as r:
        return r.read()

items = json.loads(get(API))
cands = [(i["date"], i["source_url"]) for i in items
         if re.search(r"desempe(n|ñ)o", i["source_url"], re.I) and "anexo" in i["source_url"].lower() and i["source_url"].lower().endswith(".xlsx")]
if not cands:
    raise RuntimeError("No MEM annex workbook found in the media index")
pub_date, url = max(cands)
name = url.rsplit("/", 1)[-1]
print(f"Newest annex: {name} (published {pub_date[:10]})")

def flag(downloaded):
    # Read by the scheduled Job's condition task: downstream layers run only when a new file landed
    dbutils.jobs.taskValues.set(key="downloaded", value=downloaded)

existing = {f.name: f.size for f in dbutils.fs.ls(LANDING)}
if name in existing:
    flag(False)
    dbutils.notebook.exit(json.dumps({"status": "SKIPPED", "reason": "already landed", "file": name, "published": pub_date[:10]}))

data = get(url, timeout=180)
digest = hashlib.sha256(data).hexdigest()
for f, size in existing.items():
    if size == len(data):
        with open(LANDING + f, "rb") as fh:
            if hashlib.sha256(fh.read()).hexdigest() == digest:
                flag(False)
                dbutils.notebook.exit(json.dumps({"status": "SKIPPED", "reason": f"same content as {f}", "file": name, "published": pub_date[:10]}))

wb = openpyxl.load_workbook(io.BytesIO(data), read_only=True)
missing = REQUIRED_SHEETS - set(wb.sheetnames)
if missing:
    raise ValueError(f"Downloaded file failed the structural contract, missing sheets: {sorted(missing)}")
with open(LANDING + name, "wb") as fh:
    fh.write(data)
flag(True)
dbutils.notebook.exit(json.dumps({"status": "DOWNLOADED", "file": name, "published": pub_date[:10], "bytes": len(data), "sha256": digest[:16]}))